In [4]:
import lamindb as ln
import scanpy as sc

→ connected lamindb: anonymous/lamin_storage


### Initialized a local LaminDB storage folder

In [39]:
ln.setup.init(storage="./lamin_storage")

→ doing nothing, already connected lamindb: anonymous/lamin_storage
→ connected lamindb: anonymous/lamin_storage


### Started tracking my notebook run

In [ ]:
ln.track()

### Downloaded the PBMC3k dataset

In [ ]:
adata = sc.datasets.pbmc3k()
adata.write("fundamental_data_structures_and_frameworks.h5ad")

### Saved it as an AnnData artifact 

In [ ]:
af = ln.Artifact(
    "fundamental_data_structures_and_frameworks.h5ad",
    key="introduction/fundamental_data_structures_and_frameworks.h5ad",
    description="anndata for fundamental data structures and frameworks",
).save()
af

### Initializing an AnnData object

In [9]:
import warnings

warnings.filterwarnings("ignore")


In [10]:
import anndata as ad
import lamindb as ln
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

ln.track()

→ loaded Transform('aV9wcZ2u9C310000', key='fundamental_data_structures_and_frameworks_practice.ipynb'), re-started Run('TtJEhrfbHik5VJVP') at 2026-03-13 17:58:07 UTC
→ notebook imports: anndata==0.12.7 lamindb==2.2.1 numpy==2.4.2 pandas==2.3.3 scanpy==1.12 scipy==1.16.3
• recommendation: to identify the notebook across renames, pass the uid: ln.track("aV9wcZ2u9C31")


As a next step, we initialize an AnnData object with random {term}Poisson distributed <Poisson distribution> data. It is an unwritten rule to name the primary AnnData object of the analysis adata

In [ ]:
counts = csr_matrix(
    np.random.default_rng().poisson(1, size=(100, 2000)), dtype=np.float32
)
adata = ad.AnnData(counts)
adata

The obtained AnnData object has 100 observations and 2000 variables. This would correspond to 100 cells with 2000 genes. 
The initial data we passed are accessible as a sparse matrix using adata.X.

In [41]:
adata.X

CSRDataset: backend hdf5, shape (100, 2000), data_dtype float32

Now, we provide the index to both the obs and var axes using .obs_names and .var_names, respectively.

In [42]:
adata.obs_names = [f"Cell_{i:d}" for i in range(adata.n_obs)]
adata.var_names = [f"Gene_{i:d}" for i in range(adata.n_vars)]
print(adata.obs_names[:10])

Index(['Cell_0', 'Cell_1', 'Cell_2', 'Cell_3', 'Cell_4', 'Cell_5', 'Cell_6',
       'Cell_7', 'Cell_8', 'Cell_9'],
      dtype='object')


### Adding aligned metadata

#### Observational or variable level

The core of our AnnData object is now in place. As a next step, we add metadata at both the observational and variable levels. Remember, we store such annotations in the .obs and .var slots of the AnnData object for cell and gene annotations, respectively.

In [45]:
ct = np.random.default_rng().choice(["B", "T", "Monocyte"], size=(adata.n_obs,))
adata.obs["cell_type"] = pd.Categorical(ct)  # Categoricals are preferred for efficiency
adata.obs

,cell_type
Cell_0,Monocyte
Cell_1,Monocyte
Cell_2,T
Cell_3,T
Cell_4,T
...,...
Cell_95,Monocyte
Cell_96,B
Cell_97,B
Cell_98,B


In [15]:
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'

#### Subsetting using metadata

In [16]:
# Subsetting using metadata
# We can also subset the AnnData object with the randomly generated cell types.
# The slicing and masking of the AnnData object behaves similarly to the data access in Pandas DataFrames or R matrices. 
bdata = adata[adata.obs.cell_type == "B"]
bdata

View of AnnData object with n_obs × n_vars = 44 × 2000
    obs: 'cell_type'

In [17]:
# Observation/variable-level matrices
# We might also have metadata at either level with many dimensions, such as a UMAP embedding of the data. AnnData has the .obsm/.varm attributes 
# for this type of metadata. We use keys to identify the different matrices we insert. The restriction of .obsm/.varm is that .obsm matrices must 
# have a length equal to the number of observations as .n_obs and .varm matrices must have a length equal to .n_vars. 
# They can each independently have a different number of dimensions.
# Let us start with a randomly generated matrix that we can interpret as a UMAP embedding of the data we would like to store,
# as well as some random gene-level metadata.
adata.obsm["X_umap"] = np.random.default_rng().normal(0, 1, size=(adata.n_obs, 2))
adata.varm["gene_stuff"] = np.random.default_rng().normal(0, 1, size=(adata.n_vars, 5))
adata.obsm

AxisArrays with keys: X_umap

In [18]:
# Again, the AnnData representation is updated.
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    obsm: 'X_umap'
    varm: 'gene_stuff'

In [19]:
# Unstructured metadata
# As mentioned above, AnnData has .uns, which allows for any unstructured metadata. 
# This can be anything, like a list or a dictionary, with some general information that was useful in the analysis of our data. 
# Try only using this slot for data that cannot be efficiently stored in the other slots.
adata.uns["random"] = [1, 2, 3]
adata.uns

OrderedDict([('random', [1, 2, 3])])

In [23]:
# Layers
# Finally, we may have different forms of our original core data, perhaps one that is normalized and one that is not. 
# These can be stored in different layers in AnnData. For example, let us log transform the original data and store it in a layer.
adata.layers["log_transformed"] = np.log1p(adata.X)
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    uns: 'random'
    obsm: 'X_umap'
    varm: 'gene_stuff'
    layers: 'log_transformed'

In [25]:
# Conversion to DataFrames
# It is possible to obtain a Pandas DataFrame from one of the layers.
adata.to_df(layer="log_transformed")

,Gene_0,Gene_1,Gene_2,Gene_3,Gene_4,Gene_5,Gene_6,Gene_7,Gene_8,Gene_9,...,Gene_1990,Gene_1991,Gene_1992,Gene_1993,Gene_1994,Gene_1995,Gene_1996,Gene_1997,Gene_1998,Gene_1999
Cell_0,1.386294,0.000000,0.693147,0.693147,0.693147,0.693147,1.098612,0.693147,1.098612,0.693147,...,0.000000,1.098612,0.000000,0.000000,0.693147,0.693147,0.000000,0.693147,0.693147,1.098612
Cell_1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.693147,0.693147,1.098612,0.693147,...,1.609438,0.693147,1.386294,1.098612,0.693147,0.693147,0.000000,0.693147,1.098612,1.386294
Cell_2,0.000000,1.386294,0.000000,0.000000,0.000000,0.693147,0.000000,0.693147,0.000000,0.000000,...,0.000000,0.693147,0.693147,0.693147,0.693147,1.098612,0.693147,0.693147,1.098612,0.000000
Cell_3,1.098612,0.693147,1.098612,0.000000,1.098612,0.693147,1.386294,0.693147,0.693147,0.000000,...,1.386294,0.693147,0.693147,0.693147,0.000000,0.693147,1.386294,0.000000,0.000000,0.693147
Cell_4,0.000000,0.000000,0.693147,1.098612,0.693147,1.098612,0.000000,0.000000,0.693147,1.386294,...,0.000000,1.098612,0.000000,0.000000,0.000000,1.098612,1.098612,1.098612,0.000000,0.693147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Cell_95,0.693147,0.000000,0.693147,1.386294,0.693147,0.000000,0.000000,1.098612,1.098612,0.693147,...,0.000000,0.000000,0.693147,0.000000,0.693147,0.000000,1.098612,0.693147,0.000000,0.693147
Cell_96,0.693147,0.693147,1.098612,0.693147,0.000000,0.000000,0.000000,1.098612,0.000000,1.098612,...,0.693147,0.000000,0.693147,0.000000,0.000000,1.098612,0.693147,0.693147,0.000000,0.000000
Cell_97,0.693147,0.693147,0.693147,1.098612,1.098612,0.693147,0.000000,0.000000,1.098612,0.000000,...,0.693147,0.693147,0.000000,0.000000,0.693147,0.693147,0.000000,1.098612,1.098612,0.693147
Cell_98,0.000000,1.098612,0.693147,0.693147,0.000000,0.000000,1.098612,0.693147,0.693147,1.098612,...,0.000000,1.098612,0.693147,0.693147,1.791759,0.693147,0.693147,1.098612,0.693147,0.693147


In [27]:
# Reading and writing of AnnData objects
# AnnData objects can be saved on disk to hierarchical array stores like [HDF5] or [Zarr]to enable similar structures in disk and on memory.
# AnnData comes with its own persistent HDF5-based file format: `h5ad`.
# If string columns with a few categories are not yet categorical, AnnData will auto-transform them to categorical. 
# We will now save our AnnData object in `h5ad` format.
adata.write("my_results.h5ad", compression="gzip")
adata_new = ad.read_h5ad("my_results.h5ad")
adata_new

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    uns: 'random'
    obsm: 'X_umap'
    varm: 'gene_stuff'
    layers: 'log_transformed'

In [28]:
# Efficient data access
# For the fun of it, let us look at another metadata use case.
# Imagine that the observations come from instruments characterizing 10 readouts in a multi-year study with samples 
# taken from different subjects at different sites.
# We would typically get that information in some format and then store it in a DataFrame:
obs_meta = pd.DataFrame(
    {
        "time_yr": np.random.default_rng().choice([0, 2, 4, 8], adata.n_obs),
        "subject_id": np.random.default_rng().choice(
            ["subject 1", "subject 2", "subject 4", "subject 8"], adata.n_obs
        ),
        "instrument_type": np.random.default_rng().choice(
            ["type a", "type b"], adata.n_obs
        ),
        "site": np.random.default_rng().choice(["site x", "site y"], adata.n_obs),
    },
    index=adata.obs.index,  # these are the same IDs of observations as above!
)

In [29]:
# This is how we join the readout data with the metadata. Of course, the first argument of the following call for X could also just be a DataFrame. 
# This will result in a single data container that tracks everything.
adata = ad.AnnData(adata.X, obs=obs_meta, var=adata.var)
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [30]:
# Similar to numpy arrays, AnnData objects can either hold actual data or reference another `AnnData` object.
# In the latter case, they are referred to as "view".
# Subsetting AnnData objects always returns views, which has two advantages:

# - No new memory is allocated.
# - It is possible to modify the underlying AnnData object.
 
# You can get an actual AnnData object from a view by calling `.copy()` on the view.
# Usually, this is not necessary, as any modification of elements of a view (calling `.[]` on an attribute of the view) internally calls `.copy()` and makes the view an AnnData object that holds actual data.
# See the example below.
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [31]:
# Indexing into AnnData will assume that integer arguments to `[]` behave like `.iloc` in pandas, whereas string arguments behave like `.loc`.
# `AnnData` always assumes string indices.
adata_view = adata[:5, ["Gene_1", "Gene_3"]]
adata_view

View of AnnData object with n_obs × n_vars = 5 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [32]:
# The dimensions of the AnnData object have not changed. It still contains the same data.
# If we want an AnnData that holds the data in memory, we must call it `.copy()`.
adata_subset = adata[:5, ["Gene_1", "Gene_3"]].copy()
adata_subset

AnnData object with n_obs × n_vars = 5 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [33]:
# Partial reading of large data
# If a single h5ad file is very large, you can partially read it into memory by using backed mode.
adata = ad.read_h5ad("my_results.h5ad", backed="r")

In [34]:
# If you do this, you will need to remember that the AnnData object has an open connection to the file used for reading.
adata.filename

PosixPath('my_results.h5ad')

In [35]:
adata.file.close()

In [38]:
# Unimodal data analysis with scanpy
# More specifically, scanpy is a Python package that builds on top of AnnData to facilitate the analysis of single-cell gene expression data.